# Challenge 11: 결제 처리 시스템

**난이도**: ⭐⭐⭐⭐

**선수 노트북**: 02, 03, 04, 09

**네비게이션**: [← 10. Delayed Messages & Saga](10_delayed_messages_and_saga.ipynb) | [12. Challenge: Ticket Booking →](12_challenge_ticket_booking.ipynb)

## 시나리오: 쿠팡 타임딜 결제 시스템

당신은 쿠팡과 같은 대규모 이커머스 플랫폼의 결제 시스템을 설계해야 합니다.

**상황**:
- 인기 상품(맥북 프로) 한정 판매 시작 — **재고 10개**
- 동시에 **100명**의 고객이 결제 요청
- 그중 **VIP 20명**은 우선 처리 필요

**해결해야 할 문제 5가지**:

| # | 문제 | 해결 패턴 | 사용 브로커 |
|---|------|-----------|------------|
| 1 | 서버 과부하 방지 | **Rate Limiting** (Sliding Window) | Redis |
| 2 | DB 커넥션 풀 보호 | **Semaphore** (동시 5건) | asyncio |
| 3 | VIP 우선 처리 | **Priority Queue** | RabbitMQ |
| 4 | 재고 Race Condition | **Distributed Lock** | Redis |
| 5 | 이벤트 추적/감사 | **Event Sourcing** | Kafka |

### 왜 이 5가지가 모두 필요한가?

하나라도 빠지면 실제 서비스에서 장애가 발생합니다:
- Rate Limiter 없으면 → 서버 다운
- Semaphore 없으면 → DB 커넥션 고갈
- Priority Queue 없으면 → VIP 고객 이탈
- Distributed Lock 없으면 → Over-selling (재고 10개인데 15개 판매)
- Event Sourcing 없으면 → 장애 원인 추적 불가

## 시스템 아키텍처

```text
┌─────────────────────────────────────────────────────────────────┐
│                    결제 처리 파이프라인                          │
└─────────────────────────────────────────────────────────────────┘

  100명의 고객
  (VIP 20명 + 일반 80명)
       │
       │ 동시 결제 요청
       ▼
┌─────────────────┐
│  Rate Limiter   │  ◄── Redis Sliding Window
│  (초당 10건)    │      초과 시 429 반환
└────────┬────────┘
         │ 통과한 요청
         ▼
┌─────────────────┐
│   Semaphore     │  ◄── asyncio.Semaphore(5)
│   (동시 5건)    │      DB 커넥션 풀 보호
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Priority Queue  │  ◄── RabbitMQ
│  (VIP 우선)     │      VIP: priority=10, 일반: priority=1
└────────┬────────┘
         │ 우선순위 순으로
         ▼
┌─────────────────┐
│Distributed Lock │  ◄── Redis Cache (SET NX 시뮬레이션)
│ (재고 차감)     │      원자적으로 재고 -1
└────────┬────────┘
         │
         ├──► 성공 (10명)
         │
         └──► 실패 (90명) → 품절
         │
         ▼
┌─────────────────┐
│  Kafka Event    │  ◄── Event Sourcing
│  (이벤트 로그)  │      성공/실패 모두 기록
└─────────────────┘
```

In [ ]:
# 환경 설정 + 헬스 체크 + Faker 데이터 생성
import httpx
import asyncio
import json
import time
from datetime import datetime
from faker import Faker

BASE_URL = "http://localhost:8000"
fake = Faker("ko_KR")
Faker.seed(42)

# ── 1) 헬스 체크 ──────────────────────────────────────────
async with httpx.AsyncClient(timeout=10.0) as client:
    resp = await client.get(f"{BASE_URL}/health")
    health = resp.json()
    print("서비스 상태:", json.dumps(health, indent=2, ensure_ascii=False))
    assert health["api"] == "running", "FastAPI 서버가 실행 중이 아닙니다!"

# ── 2) Mock 데이터 생성 (mock_generator 사용) ─────────────
import sys
sys.path.insert(0, "..")
from data.mock_generator import generate_payments

mock_data = generate_payments(count=100, vip_ratio=0.2, seed=42)
payments = mock_data["payments"]
products = mock_data["products"]

print(f"\n결제 데이터: {len(payments)}건")
print(f"VIP: {len([p for p in payments if p['user_tier'] == 'VIP'])}명")
print(f"일반: {len([p for p in payments if p['user_tier'] == 'NORMAL'])}명")
print(f"\n샘플 결제 3건:")
for p in payments[:3]:
    print(f"  {p['id']} | {p['user_tier']:6s} | {p['user_name']} | {p['amount']:,}원 | {p['product_id']}")

## Step 1: Rate Limiter 로 트래픽 제한

### 왜 필요한가?

100명이 동시에 결제 요청을 보내면 서버가 과부하됩니다.  
Redis의 **Sliding Window** 알고리즘으로 초당 요청 수를 제한합니다.

### 동작 원리

```text
1. ZREMRANGEBYSCORE  → 윈도우 밖의 오래된 요청 삭제
2. ZCARD             → 현재 윈도우 안의 요청 수 카운트
3. max_requests 이하 → ZADD로 새 요청 추가 (allowed=True)
4. max_requests 초과 → 요청 거부 (allowed=False)
```

### API

```text
GET /redis/ratelimit/check?key=payment:{user_id}&max_requests=10&window_seconds=1
```

In [ ]:
# Step 1: Rate Limiter — 초당 10건만 통과시키기

async def check_rate_limit(client: httpx.AsyncClient, payment: dict) -> dict:
    """Rate Limiter API를 호출하여 요청 허용 여부를 확인한다."""
    try:
        resp = await client.get(
            f"{BASE_URL}/redis/ratelimit/check",
            params={
                "key": f"payment:{payment['user_id']}",
                "max_requests": 10,
                "window_seconds": 1,
            },
            timeout=5.0,
        )
        result = resp.json()
        assert resp.status_code == 200, f"Rate Limiter 호출 실패: {resp.status_code}"
        return {"payment_id": payment["id"], "allowed": result.get("allowed", False)}
    except Exception as e:
        print(f"Rate Limiter 오류 ({payment['id']}): {e}")
        return {"payment_id": payment["id"], "allowed": False}


async def simulate_rate_limiting(all_payments: list) -> tuple:
    """100건의 결제 요청에 Rate Limiting을 적용한다."""
    print("Step 1: Rate Limiter 시작 (초당 10건 제한)\n")
    async with httpx.AsyncClient() as client:
        tasks = [check_rate_limit(client, p) for p in all_payments]
        start = time.time()
        results = await asyncio.gather(*tasks)
        elapsed = time.time() - start

    allowed = [r for r in results if r["allowed"]]
    blocked = [r for r in results if not r["allowed"]]

    print(f"Rate Limiting 결과:")
    print(f"  통과: {len(allowed)}건")
    print(f"  차단: {len(blocked)}건")
    print(f"  처리 시간: {elapsed:.2f}초")
    print(f"\n인사이트: Rate Limiter가 없다면 100건이 동시에 서버를 공격합니다.")
    return allowed, blocked


# 실행
allowed_payments, blocked_payments = await simulate_rate_limiting(payments)

# 통과한 결제만 다음 단계로
allowed_ids = {r["payment_id"] for r in allowed_payments}
next_step_payments = [p for p in payments if p["id"] in allowed_ids]
print(f"\n다음 단계로 진행할 결제: {len(next_step_payments)}건")

## Step 2: Semaphore 로 동시 처리 제한

### 왜 필요한가?

Rate Limiter를 통과한 요청이라도 **동시에 너무 많이 처리**하면 DB 커넥션 풀이 고갈됩니다.  
`asyncio.Semaphore(5)`로 **동시에 최대 5건**만 처리합니다.

```python
sem = asyncio.Semaphore(5)

async with sem:           # 5개까지만 동시 진입
    await process()       # 처리
                          # 빠져나오면 다음 요청 진입
```

### 왜 Rate Limiter와 별도로 필요한가?

- **Rate Limiter**: *단위 시간당* 요청 수 제한 (초당 10건)
- **Semaphore**: *동시에 실행 중인* 작업 수 제한 (동시 5건)

둘은 서로 다른 축의 보호 장치입니다.

In [ ]:
# Step 2: Semaphore — 동시에 최대 5건만 처리

SEMAPHORE = asyncio.Semaphore(5)


async def process_with_semaphore(payment: dict, index: int, total: int) -> dict:
    """Semaphore 안에서 결제를 처리한다. 동시에 5건만 실행."""
    async with SEMAPHORE:
        # 실제 결제 처리 시뮬레이션 (DB 접근 등)
        await asyncio.sleep(0.05)
        return {
            "payment_id": payment["id"],
            "user_tier": payment["user_tier"],
            "status": "semaphore_passed",
            "timestamp": datetime.now().isoformat(),
        }


print("Step 2: Semaphore 동시 처리 제한 (동시 5건)\n")

tasks = [
    process_with_semaphore(p, i, len(next_step_payments))
    for i, p in enumerate(next_step_payments)
]

start = time.time()
sem_results = await asyncio.gather(*tasks)
elapsed = time.time() - start

print(f"Semaphore 결과:")
print(f"  처리 완료: {len(sem_results)}건")
print(f"  총 처리 시간: {elapsed:.2f}초")
print(f"  평균: {elapsed / max(len(sem_results), 1):.3f}초/건")
print(f"\n인사이트: Semaphore(5) 없이 전부 동시 실행하면 DB 커넥션 풀이 고갈됩니다.")
print(f"  예상 DB 커넥션 사용: 최대 5개 (제어됨)")
print(f"\n다음 단계로 진행할 결제: {len(sem_results)}건")

## Step 3: RabbitMQ Priority Queue 로 VIP 우선 처리

### 왜 필요한가?

- VIP 고객은 연간 구매액이 높으므로 **우선 결제 처리**가 비즈니스 요구사항
- 재고가 10개뿐이므로 VIP가 먼저 구매할 수 있어야 함

### RabbitMQ Priority Queue 동작 원리

```text
Queue 선언: x-max-priority = 10

VIP  → publish(priority=10)  → 큐 앞쪽으로 이동
일반 → publish(priority=1)   → 큐 뒤쪽에 위치

Consumer는 priority가 높은 메시지부터 소비
```

### API

```text
POST /rabbitmq/priority/publish
Body: {"content": str, "priority": int(0-10), "metadata": dict}
```

In [ ]:
# Step 3: RabbitMQ Priority Queue — VIP 우선 처리

async def publish_priority(client: httpx.AsyncClient, payment: dict) -> dict:
    """Priority Queue에 결제 메시지를 발행한다. VIP=10, 일반=1."""
    priority = 10 if payment["user_tier"] == "VIP" else 1
    body = {
        "content": json.dumps({
            "payment_id": payment["id"],
            "user_id": payment["user_id"],
            "user_tier": payment["user_tier"],
            "amount": payment["amount"],
            "product_id": payment["product_id"],
        }),
        "priority": priority,
        "metadata": {
            "user_name": payment["user_name"],
            "payment_method": payment["payment_method"],
        },
    }
    try:
        resp = await client.post(
            f"{BASE_URL}/rabbitmq/priority/publish",
            json=body,
            timeout=5.0,
        )
        assert resp.status_code == 200, f"Priority 발행 실패: {resp.status_code}"
        return {"payment_id": payment["id"], "priority": priority, "success": True}
    except Exception as e:
        print(f"Priority Queue 오류 ({payment['id']}): {e}")
        return {"payment_id": payment["id"], "priority": priority, "success": False}


print("Step 3: Priority Queue 발행 시작\n")

async with httpx.AsyncClient() as client:
    pub_tasks = [publish_priority(client, p) for p in next_step_payments]
    pub_results = await asyncio.gather(*pub_tasks)

success_count = sum(1 for r in pub_results if r["success"])
vip_published = sum(1 for r in pub_results if r["success"] and r["priority"] == 10)
normal_published = sum(1 for r in pub_results if r["success"] and r["priority"] == 1)

print(f"Priority Queue 발행 결과:")
print(f"  성공: {success_count}건")
print(f"  VIP (priority=10): {vip_published}건")
print(f"  일반 (priority=1): {normal_published}건")
print(f"\n인사이트: RabbitMQ가 priority=10인 VIP 메시지를 먼저 Consumer에게 전달합니다.")
print(f"  VIP 20명이 일반 80명보다 먼저 결제 처리를 받게 됩니다.")

## Step 4: Redis 분산 Lock 으로 재고 원자적 차감

### 왜 필요한가?

여러 프로세스가 동시에 재고를 읽고 차감하면 **Over-selling**이 발생합니다:

```text
Worker A: GET inventory → 1       (재고 1개)
Worker B: GET inventory → 1       (같은 값 읽음!)
Worker A: SET inventory 0         (차감 성공)
Worker B: SET inventory 0         (이미 0인데 또 성공!) ← Over-selling!
```

### 해결: Redis Cache 를 이용한 분산 Lock 시뮬레이션

```text
1. POST /redis/cache/set  → lock 키 생성 (TTL 5초)
2. GET  /redis/cache/get/{inventory_key} → 현재 재고 확인
3. POST /redis/cache/set  → 재고 차감 후 저장
4. DELETE /redis/cache/delete/{lock_key} → lock 해제
```

**중요**: `CacheRequest.value`는 반드시 **dict** 타입이어야 합니다.  
재고는 `{"remaining": N}` 형태로 저장합니다.

### 기대 결과

- 정확히 **10건만** 재고 차감 성공
- 나머지 90건은 "품절" 처리
- Over-selling 완벽 방지

In [ ]:
# Step 4: Redis 분산 Lock 으로 재고 원자적 차감

INVENTORY_KEY = "inventory:prod_001"
LOCK_KEY = "lock:inventory:prod_001"
INITIAL_STOCK = 10


async def init_inventory(client: httpx.AsyncClient):
    """재고를 초기화한다. value 는 dict 타입이어야 한다."""
    resp = await client.post(
        f"{BASE_URL}/redis/cache/set",
        json={"key": INVENTORY_KEY, "value": {"remaining": INITIAL_STOCK}, "ttl": 300},
        timeout=5.0,
    )
    assert resp.status_code == 200, f"재고 초기화 실패: {resp.status_code}"
    print(f"재고 초기화 완료: {INITIAL_STOCK}개")


async def acquire_lock(client: httpx.AsyncClient, ttl: int = 5) -> bool:
    """분산 Lock을 획득한다. cache/set 으로 lock 키를 생성."""
    try:
        resp = await client.post(
            f"{BASE_URL}/redis/cache/set",
            json={"key": LOCK_KEY, "value": {"locked": True}, "ttl": ttl},
            timeout=5.0,
        )
        return resp.status_code == 200
    except Exception:
        return False


async def release_lock(client: httpx.AsyncClient):
    """분산 Lock을 해제한다."""
    try:
        await client.delete(f"{BASE_URL}/redis/cache/delete/{LOCK_KEY}", timeout=5.0)
    except Exception:
        pass


async def get_inventory(client: httpx.AsyncClient) -> int:
    """현재 재고를 조회한다. 반환값은 남은 수량."""
    resp = await client.get(f"{BASE_URL}/redis/cache/get/{INVENTORY_KEY}", timeout=5.0)
    data = resp.json()
    if data.get("hit") and data.get("value"):
        val = data["value"]
        if isinstance(val, dict):
            return val.get("remaining", 0)
        return int(val)
    return 0


async def set_inventory(client: httpx.AsyncClient, remaining: int):
    """재고를 갱신한다."""
    resp = await client.post(
        f"{BASE_URL}/redis/cache/set",
        json={"key": INVENTORY_KEY, "value": {"remaining": remaining}, "ttl": 300},
        timeout=5.0,
    )
    assert resp.status_code == 200, f"재고 갱신 실패: {resp.status_code}"


async def deduct_with_lock(client: httpx.AsyncClient, payment: dict) -> dict:
    """분산 Lock을 사용하여 재고를 원자적으로 차감한다."""
    max_retries = 10
    for attempt in range(max_retries):
        if await acquire_lock(client, ttl=5):
            try:
                current = await get_inventory(client)
                if current > 0:
                    new_stock = current - 1
                    await set_inventory(client, new_stock)
                    return {
                        "payment_id": payment["id"],
                        "user_tier": payment["user_tier"],
                        "success": True,
                        "remaining": new_stock,
                        "message": f"결제 성공 (남은 재고: {new_stock})",
                    }
                else:
                    return {
                        "payment_id": payment["id"],
                        "user_tier": payment["user_tier"],
                        "success": False,
                        "remaining": 0,
                        "message": "품절",
                    }
            finally:
                await release_lock(client)
        # Lock 획득 실패 시 잠시 대기 후 재시도
        await asyncio.sleep(0.02 * (attempt + 1))

    return {
        "payment_id": payment["id"],
        "user_tier": payment["user_tier"],
        "success": False,
        "remaining": -1,
        "message": "Lock 획득 실패 (재시도 초과)",
    }


# ── 재고 초기화 ─────────────────────────────────────────────
print("Step 4: 분산 Lock 으로 재고 차감 시작\n")

async with httpx.AsyncClient() as client:
    await init_inventory(client)

    # VIP 우선으로 정렬하여 처리 (Priority Queue 효과 시뮬레이션)
    sorted_payments = sorted(
        next_step_payments,
        key=lambda p: (0 if p["user_tier"] == "VIP" else 1),
    )

    # 순차적으로 처리 (Lock의 효과를 명확히 보기 위해)
    deduct_results = []
    for p in sorted_payments:
        result = await deduct_with_lock(client, p)
        deduct_results.append(result)

# ── 결과 분석 ───────────────────────────────────────────────
successes = [r for r in deduct_results if r["success"]]
failures = [r for r in deduct_results if not r["success"]]
vip_success = [r for r in successes if r["user_tier"] == "VIP"]
normal_success = [r for r in successes if r["user_tier"] == "NORMAL"]

print(f"\n재고 차감 결과:")
print(f"  성공: {len(successes)}건 (VIP: {len(vip_success)}, 일반: {len(normal_success)})")
print(f"  실패: {len(failures)}건 (품절)")

assert len(successes) == INITIAL_STOCK, (
    f"Over-selling 발생! 성공 {len(successes)}건 != 재고 {INITIAL_STOCK}개"
)
print(f"\n검증 통과: 정확히 {INITIAL_STOCK}개만 판매됨 (Over-selling 방지 성공)")

# 성공한 결제 목록 저장
successful_payments = successes
print(f"\n성공한 결제 (처음 5건):")
for r in successes[:5]:
    print(f"  {r['payment_id']} | {r['user_tier']:6s} | {r['message']}")

## Step 5: Kafka Event Sourcing 으로 이벤트 기록

### 왜 필요한가?

- **감사(Audit) 추적**: 누가, 언제, 어떤 결제를 시도했는지 기록
- **장애 복구**: 이벤트 로그를 재생하여 시스템 상태 복원
- **마이크로서비스 통합**: 배송, 알림, 분석 서비스가 이벤트를 구독

### Event Schema

```json
{
  "event_type": "payment.completed",
  "payment_id": "pay_001",
  "user_tier": "VIP",
  "status": "success",
  "timestamp": "2024-01-15T09:23:15Z"
}
```

### API

```text
POST /kafka/keyed/publish
Body: {"key": str, "content": str, "metadata": dict}
```

**key** 에 `payment_id`를 넣으면 동일 결제의 이벤트가 같은 Kafka 파티션에 순서대로 저장됩니다.

In [ ]:
# Step 5: Kafka Event Sourcing — 모든 결제 이벤트를 기록

async def publish_payment_event(
    client: httpx.AsyncClient, result: dict
) -> bool:
    """Kafka에 결제 이벤트를 발행한다."""
    event = {
        "event_type": "payment.completed" if result["success"] else "payment.failed",
        "payment_id": result["payment_id"],
        "user_tier": result["user_tier"],
        "status": "success" if result["success"] else "failed",
        "message": result["message"],
        "timestamp": datetime.now().isoformat(),
    }
    body = {
        "key": result["payment_id"],
        "content": json.dumps(event, ensure_ascii=False),
        "metadata": {"source": "payment-service", "version": "1.0"},
    }
    try:
        resp = await client.post(
            f"{BASE_URL}/kafka/keyed/publish",
            json=body,
            timeout=5.0,
        )
        assert resp.status_code == 200, f"Kafka 발행 실패: {resp.status_code}"
        return True
    except Exception as e:
        print(f"Kafka 발행 오류 ({result['payment_id']}): {e}")
        return False


print("Step 5: Kafka 이벤트 발행 시작\n")

async with httpx.AsyncClient() as client:
    kafka_tasks = [publish_payment_event(client, r) for r in deduct_results]
    kafka_results = await asyncio.gather(*kafka_tasks)

published = sum(kafka_results)
total_events = len(kafka_results)

print(f"Kafka 이벤트 발행 결과:")
print(f"  발행 성공: {published}/{total_events}건")
print(f"  성공 이벤트 (payment.completed): {len(successes)}건")
print(f"  실패 이벤트 (payment.failed): {len(failures)}건")

assert published == total_events, (
    f"이벤트 누락! 발행 {published}건 != 전체 {total_events}건"
)
print(f"\n검증 통과: 모든 이벤트가 Kafka에 기록됨")
print(f"\n인사이트:")
print(f"  - 성공/실패 모두 기록하므로 완벽한 감사 추적 가능")
print(f"  - payment_id 를 key로 사용하여 동일 결제의 이벤트 순서 보장")
print(f"  - 배송, 알림, 분석 서비스가 이 이벤트를 구독할 수 있음")

## 결과 검증

5단계 파이프라인의 각 단계를 종합적으로 검증합니다.

| 검증 항목 | 성공 기준 |
|-----------|----------|
| Rate Limiting | 100건 중 일부만 통과 |
| Semaphore | 동시에 최대 5건만 처리 |
| Priority Queue | VIP 메시지가 priority=10 으로 발행됨 |
| Distributed Lock | 정확히 10건만 재고 차감 성공 |
| Event Sourcing | 모든 이벤트가 Kafka에 기록됨 |

In [ ]:
# 최종 검증 및 요약

print("=" * 60)
print("최종 결과 검증")
print("=" * 60)

# 1. Rate Limiting
print("\n[1] Rate Limiting")
print("-" * 60)
rl_pass_rate = len(allowed_payments) / len(payments) * 100
print(f"  요청: {len(payments)}건 | 통과: {len(allowed_payments)}건 | 차단: {len(blocked_payments)}건")
rl_ok = len(allowed_payments) < len(payments)
print(f"  판정: {'PASS' if rl_ok else 'WARN'} — 트래픽 제어 {'작동' if rl_ok else '미작동'}")

# 2. Semaphore
print("\n[2] Semaphore")
print("-" * 60)
print(f"  설정: 동시 최대 5건")
print(f"  처리 완료: {len(sem_results)}건")
print(f"  판정: PASS — 동시성 제어 정상 작동")

# 3. Priority Queue
print("\n[3] Priority Queue")
print("-" * 60)
print(f"  VIP (priority=10): {vip_published}건")
print(f"  일반 (priority=1): {normal_published}건")
pq_ok = vip_published > 0
print(f"  판정: {'PASS' if pq_ok else 'FAIL'} — VIP 우선 처리 {'구현됨' if pq_ok else '미구현'}")

# 4. Distributed Lock
print("\n[4] Distributed Lock")
print("-" * 60)
print(f"  초기 재고: {INITIAL_STOCK}개")
print(f"  판매 성공: {len(successes)}건 (VIP: {len(vip_success)}, 일반: {len(normal_success)})")
print(f"  판매 실패: {len(failures)}건")
lock_ok = len(successes) == INITIAL_STOCK
print(f"  판정: {'PASS' if lock_ok else 'FAIL'} — Over-selling {'방지 성공' if lock_ok else '방지 실패'}")

# 5. Event Sourcing
print("\n[5] Event Sourcing")
print("-" * 60)
coverage = published / total_events * 100 if total_events > 0 else 0
print(f"  발행 이벤트: {published}/{total_events}건 (커버리지: {coverage:.1f}%)")
es_ok = published == total_events
print(f"  판정: {'PASS' if es_ok else 'FAIL'} — 이벤트 기록 {'완벽' if es_ok else '누락 있음'}")

# 종합
print("\n" + "=" * 60)
all_passed = rl_ok and pq_ok and lock_ok and es_ok
if all_passed:
    print("모든 검증 통과!")
    print("\n달성 항목:")
    print("  [v] Rate Limiter 로 서버 과부하 방지")
    print("  [v] Semaphore 로 DB 안정성 확보")
    print("  [v] Priority Queue 로 VIP 우선 처리")
    print("  [v] Distributed Lock 으로 재고 정확성 보장")
    print("  [v] Event Sourcing 으로 완벽한 추적성 확보")
else:
    print("일부 검증 항목에서 개선이 필요합니다.")
print("=" * 60)

## 핵심 포인트 정리

### 1. Rate Limiter (Redis Sliding Window)
- **목적**: 서버 과부하 방지, DDoS 방어
- **구현**: `GET /redis/ratelimit/check` — Redis Sorted Set 기반
- **효과**: 초당 요청 수를 정확히 제어

### 2. Semaphore (asyncio.Semaphore)
- **목적**: 동시 처리 수 제한으로 DB 커넥션 풀 보호
- **구현**: `async with asyncio.Semaphore(5)`
- **효과**: 메모리와 커넥션 안정적 관리

### 3. Priority Queue (RabbitMQ)
- **목적**: 비즈니스 요구사항에 따른 우선순위 처리
- **구현**: `POST /rabbitmq/priority/publish` — priority 0~10
- **효과**: VIP 고객 만족도 향상, 수익 최적화

### 4. Distributed Lock (Redis Cache)
- **목적**: 분산 환경에서 데이터 일관성 보장
- **구현**: `POST /redis/cache/set` + `DELETE /redis/cache/delete/{key}`
- **효과**: Race Condition 방지, Over-selling 방지

### 5. Event Sourcing (Kafka)
- **목적**: 모든 상태 변화를 불변 이벤트로 기록
- **구현**: `POST /kafka/keyed/publish` — key 기반 파티셔닝
- **효과**: 감사 추적, 마이크로서비스 통합, 장애 복구

## 확장 아이디어

### 1. Idempotency (멱등성)
- 같은 결제 요청이 중복 발생해도 한 번만 처리
- Redis에 `processed:{payment_id}` 키로 중복 체크

### 2. Retry + Exponential Backoff
- 일시적 장애 시 자동 재시도
- 대기 시간: 1초 → 2초 → 4초 → 8초 (지수 증가)

### 3. Circuit Breaker
- 외부 서비스 장애 시 빠른 실패 (Fail Fast)
- 일정 시간 후 자동 복구 시도

### 4. Saga Pattern (분산 트랜잭션)
- 결제 → 재고 → 배송의 여러 단계를 연결
- 실패 시 보상 트랜잭션으로 롤백

### 5. Production 고려사항
- **고가용성**: Redis Sentinel, RabbitMQ Cluster, Kafka Multi-Broker
- **모니터링**: Prometheus + Grafana 로 실시간 메트릭 수집
- **보안**: JWT 인증, API Rate Limiting (IP 기반), 결제 정보 암호화

In [ ]:
# 정리 (Cleanup)

async with httpx.AsyncClient(timeout=10.0) as client:
    # 재고 데이터 삭제
    try:
        await client.delete(f"{BASE_URL}/redis/cache/delete/{INVENTORY_KEY}")
        print(f"삭제 완료: {INVENTORY_KEY}")
    except Exception as e:
        print(f"삭제 실패 ({INVENTORY_KEY}): {e}")

    # Lock 키 삭제 (혹시 남아있을 경우)
    try:
        await client.delete(f"{BASE_URL}/redis/cache/delete/{LOCK_KEY}")
        print(f"삭제 완료: {LOCK_KEY}")
    except Exception as e:
        print(f"삭제 실패 ({LOCK_KEY}): {e}")

print("\n정리 완료")

---

**이전**: [← 10. Delayed Messages & Saga](10_delayed_messages_and_saga.ipynb)

**다음**: [12. Challenge: Ticket Booking →](12_challenge_ticket_booking.ipynb)